# Quick model test -> full cluster job

Workflow:
1. Prototype here on a cheap **1x L4** notebook (pick the *GPU dev - 1x L4* profile at spawn).
2. When it works, submit the real multi-node job to the **H200 Slurm** cluster.

Your home dir is the cluster's shared NFS, so files saved here are visible to Slurm jobs.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# Tiny single-GPU training step to confirm the stack end-to-end.
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Sequential(nn.Linear(1024, 4096), nn.ReLU(), nn.Linear(4096, 1024)).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(256, 1024, device=dev)
for step in range(20):
    opt.zero_grad()
    loss = model(x).pow(2).mean()
    loss.backward(); opt.step()
print('final loss', float(loss))

## Submit the full multi-node job to Slurm

The `sbatch`/`squeue` commands run on the cluster login node over SSH (configured in the image). The job script lives on the shared NFS home, so the path is identical cluster-side.

In [ ]:
# Copy the example job into your home (shared NFS), then submit it.
!cp -n /opt/examples/sbatch-2node-ddp.sh ~/ 2>/dev/null || true
!sbatch ~/sbatch-2node-ddp.sh
!squeue

In [ ]:
# Tail the most recent job output (written to your shared home).
!ls -t ~/ddp-*.out 2>/dev/null | head -1 | xargs -r tail -n 40